In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
#GENERATE DATA
np.random.seed(42)
torch.manual_seed(42)
N_USERS = 10
N_DAYS = 30

def generate_health_data():
    user_ids = [f"User_{i:02d}" for i in range(1, N_USERS + 1)]
    ages = np.random.randint(18, 80, N_USERS)
    bmis = np.random.uniform(18.0, 35.0, N_USERS)
    profiles = pd.DataFrame({'User_ID': user_ids, 'Age': ages, 'BMI': bmis})

    records = []
    for _, user in profiles.iterrows():
        uid, age, bmi = user['User_ID'], user['Age'], user['BMI']
        base_hr = 50 + ((age - 18) * 0.15) + ((bmi - 18) * 1.0)
        base_bpsys = 100 + (age * 0.3) + (bmi * 0.8)
        base_bpdia = 65 + (age * 0.15) + (bmi * 0.5)
        base_sugar = 75 + (bmi * 0.7) + (age * 0.1)
        base_water = 2000
        base_steps = 10000 - (age * 30) - (bmi * 50)

        for day in range(1, N_DAYS + 1):
            anomaly_mult = 4 if np.random.rand() < 0.05 else 1

            records.append({
                'User_ID': uid, 'Day': day,
                'HR_avg': base_hr + np.random.normal(0, 4 * anomaly_mult),
                'BP_sys': base_bpsys + np.random.normal(0, 5 * anomaly_mult),
                'BP_dia': base_bpdia + np.random.normal(0, 4 * anomaly_mult),
                'Blood_Sugar': base_sugar + np.random.normal(0, 8 * anomaly_mult),
                'Water_intake': base_water + np.random.normal(0, 400 * anomaly_mult),
                'Steps': base_steps + np.random.normal(0, 1500 * anomaly_mult)
            })
    return pd.DataFrame(records)

df = generate_health_data()
# PREPROCESSUING
features = ['HR_avg', 'BP_sys', 'BP_dia', 'Blood_Sugar', 'Water_intake', 'Steps']

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])
X_tensor = torch.FloatTensor(scaled_data)

# BUILDING THE AUTOENCODER
class HealthAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(HealthAutoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 4),
            nn.ReLU(),
            nn.Linear(4, 2)
        )

        self.decoder = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU(),
            nn.Linear(4, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

model = HealthAutoencoder(input_dim=len(features))
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# TRAINING
print("Training Autoencoder...")
EPOCHS = 150

for epoch in range(EPOCHS):
    optimizer.zero_grad()

    reconstructed = model(X_tensor)
    loss = criterion(reconstructed, X_tensor)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {loss.item():.4f}")

# INFERENCE
model.eval()
with torch.no_grad():
    predictions = model(X_tensor)
    reconstruction_errors = torch.mean((predictions - X_tensor)**2, dim=1).numpy()

df['Reconstruction_Error'] = reconstruction_errors
THRESHOLD = np.percentile(df['Reconstruction_Error'], 95)
df['Is_Anomaly'] = df['Reconstruction_Error'] > THRESHOLD


print("-" * 50)
print(f"Cutoff Threshold set at: {THRESHOLD:.4f}")
print("-" * 50)

total_anomalies = df['Is_Anomaly'].sum()
print(f"TOTAL ANOMALIES DETECTED: {total_anomalies} out of {len(df)} days\n")

print("--- ANOMALIES PER USER ---")
for uid in sorted(df['User_ID'].unique()):
    user_data = df[df['User_ID'] == uid]
    anomalies = user_data[user_data['Is_Anomaly']]

    if not anomalies.empty:
        day_info = [f"Day {r['Day']} (Error: {r['Reconstruction_Error']:.2f})" for _, r in anomalies.iterrows()]
        print(f"[{uid}] {len(anomalies)} Anomalies | {', '.join(day_info)}")
    else:
        print(f"[{uid}] 0 Anomalies | User remained stable.")

Training Autoencoder...
Epoch [50/150], Loss: 0.8108
Epoch [100/150], Loss: 0.6209
Epoch [150/150], Loss: 0.5836
--------------------------------------------------
Cutoff Threshold set at: 1.6301
--------------------------------------------------
TOTAL ANOMALIES DETECTED: 15 out of 300 days

--- ANOMALIES PER USER ---
[User_01] 2 Anomalies | Day 14 (Error: 5.09), Day 23 (Error: 4.84)
[User_02] 2 Anomalies | Day 6 (Error: 5.64), Day 24 (Error: 5.05)
[User_03] 0 Anomalies | User remained stable.
[User_04] 4 Anomalies | Day 8 (Error: 1.88), Day 14 (Error: 5.84), Day 15 (Error: 6.19), Day 24 (Error: 4.29)
[User_05] 0 Anomalies | User remained stable.
[User_06] 2 Anomalies | Day 8 (Error: 3.42), Day 26 (Error: 8.56)
[User_07] 0 Anomalies | User remained stable.
[User_08] 0 Anomalies | User remained stable.
[User_09] 3 Anomalies | Day 6 (Error: 4.12), Day 13 (Error: 3.17), Day 27 (Error: 4.56)
[User_10] 2 Anomalies | Day 4 (Error: 2.23), Day 17 (Error: 2.29)
